# Sentiment Analysis: Exploratory Data Analysis & Text Representation

This notebook combines Exploratory Data Analysis (EDA) on the Stanford IMDB movie review dataset and demonstrates two fundamental text representation techniques:
1. **Bag of Words (BoW)** (both a custom implementation from scratch and Scikit-Learn's `CountVectorizer`)
2. **TF-IDF (Term Frequency-Inverse Document Frequency)** (Scikit-Learn's `TfidfVectorizer`)

---

## 1. Setup & Loading the Dataset

We will start by loading the standard IMDB dataset from Hugging Face's `datasets` library. This dataset contains 25,000 training reviews, 25,000 testing reviews, and 50,000 unsupervised reviews.

In [1]:
from datasets import load_dataset

# Load the IMDB dataset
dataset = load_dataset("stanfordnlp/imdb")
print(dataset)

c:\Users\spide\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


Let's inspect the first review in the training set along with the schema (features) of the dataset.

In [2]:
# View the first training sample
print("Sample Review:")
print(dataset["train"][0])

print("\nDataset Features:")
print(dataset["train"].features)

Sample Review:
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are

## 2. Exploratory Data Analysis (EDA)

First, let's check the distribution of the class labels to see if the dataset is balanced. The label `0` represents a negative review, and `1` represents a positive review.

In [3]:
from collections import Counter

train_labels = dataset["train"]["label"]
print("Label distribution in train set:", Counter(train_labels))

Label distribution in train set: Counter({0: 12500, 1: 12500})


Now, let's analyze the lengths of the reviews. We will inspect both:
1. **Character counts**: The raw number of characters in each review.
2. **Word counts**: The number of space-separated words in each review.

In [4]:
# Analyze character lengths of reviews
review_lengths = [len(review["text"]) for review in dataset["train"]]

print("First 5 review character lengths:", review_lengths[:5])
print("Minimum length (chars):", min(review_lengths))
print("Maximum length (chars):", max(review_lengths))
print("Average length (chars):", sum(review_lengths) / len(review_lengths))

First 5 review character lengths: [1640, 1294, 528, 706, 1814]
Minimum length (chars): 52
Maximum length (chars): 13704
Average length (chars): 1325.06964


In [5]:
# Analyze word counts of reviews
review_words = [len(review["text"].split()) for review in dataset["train"]]

print("First 10 review word counts:", review_words[:10])
print("Total number of training reviews:", len(review_words))
print("Minimum words:", min(review_words))
print("Maximum words:", max(review_words))
print("Average words:", sum(review_words) / len(review_words))

First 10 review word counts: [288, 214, 93, 118, 311, 123, 114, 301, 480, 224]
Total number of training reviews: 25000
Minimum words: 10
Maximum words: 2470
Average words: 233.7872


## 3. Text Preprocessing

Before converting text to numerical features, we need to clean and preprocess it. Raw text contains punctuation and common stopwords (like "the", "is", "and") that do not contribute much to the sentiment.

We will build a preprocessing pipeline that:
1. **Removes punctuation** (using `string.punctuation`)
2. **Converts to lowercase**
3. **Removes stopwords** (using NLTK's stopword list)

In [6]:
import string

print("Punctuation characters to remove:")
print(string.punctuation)

Punctuation characters to remove:
!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


In [7]:
# Custom function to remove punctuation
def remove_punctuation(text):
    cleaned_text = ""
    for char in text:
        if char not in string.punctuation:
            cleaned_text += char
    return cleaned_text

sample_review = dataset["train"][0]["text"]
print("Original Review (Truncated):")
print(sample_review[:300])

print("\nCleaned Review (Truncated):")
print(remove_punctuation(sample_review)[:300])

Original Review (Truncated):
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h

Cleaned Review (Truncated):
I rented I AM CURIOUSYELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967 I also heard that at first it was seized by US customs if it ever tried to enter this country therefore being a fan of films considered controversial I really had to s


In [8]:
import nltk
# Download stopwords list
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\spide\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [9]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

# Custom function to remove stopwords
def remove_stopwords(text):
    words = text.split()
    filtered = [word for word in words if word.lower() not in stop_words]
    return " ".join(filtered)

print("Sample stopword removal test:")
print(remove_stopwords("This movie is really amazing and I love it"))

Sample stopword removal test:
movie really amazing love


In [10]:
# Unified preprocessing function
def preprocess_text(text):
    # Lowercase the text
    text = text.lower()
    # Remove punctuation
    text = remove_punctuation(text)
    # Remove stopwords
    text = remove_stopwords(text)
    return text

test_input = "This MOVIE was AMAZING!!! I really loved it."
print("Original:", test_input)
print("Preprocessed:", preprocess_text(test_input))

Original: This MOVIE was AMAZING!!! I really loved it.
Preprocessed: movie amazing really loved


## 4. Text Representation: Bag of Words (BoW) from Scratch

Machine learning models require numerical input. The simplest way to represent text is **Bag of Words (BoW)**.
In BoW:
1. We construct a vocabulary of all unique words from a corpus.
2. We represent each document as a vector where each index corresponds to a vocabulary word, and the value is the frequency of that word in the document.

Let's implement this manually on a small subset of the training dataset.

In [11]:
# Select the first 20 training reviews
raw_reviews = dataset["train"]["text"][:20]

# Preprocess the reviews
reviews = [preprocess_text(review) for review in raw_reviews]

In [12]:
# Build vocabulary from processed reviews
def build_vocab(reviews_list):
    vocab = set()
    for text in reviews_list:
        words = text.split()
        for word in words:
            vocab.add(word)
    return vocab

vocab = build_vocab(reviews)
print(f"Number of unique words in the preprocessed vocabulary: {len(vocab)}")

Number of unique words in the preprocessed vocabulary: 1381


In [13]:
# Convert a single review to a vector representing word counts
def review_to_vector(review, vocabulary):
    words = review.split()
    # Ensure vocabulary is a list to preserve ordering
    vocab_list = list(vocabulary)
    vector = [words.count(word) for word in vocab_list]
    return vector

sample_text = "good movie good love"
# Let's test with a small vocabulary for visualization
test_vocab = ['good', 'bad', 'movie', 'love']
print("Test Review Vector:", review_to_vector(sample_text, test_vocab))

Test Review Vector: [2, 0, 1, 1]


In [14]:
# Convert multiple reviews to a document-term matrix
def reviews_to_matrix(reviews_list, vocabulary):
    matrix = []
    for review in reviews_list:
        matrix.append(review_to_vector(review, vocabulary))
    return matrix

sample_docs = [
    "good movie",
    "bad movie",
    "good love movie"
]
print("Document-Term Matrix:")
print(reviews_to_matrix(sample_docs, test_vocab))

Document-Term Matrix:
[[1, 0, 1, 0], [0, 1, 1, 0], [1, 0, 1, 1]]


## 5. Text Representation with Scikit-Learn

While implementing BoW from scratch is a great exercise, library implementations like Scikit-Learn's `CountVectorizer` and `TfidfVectorizer` are highly optimized and support additional options like n-grams, min/max document frequencies, and custom tokenizers.

### 5.1 Bag of Words (BoW) with `CountVectorizer`

Let's see how `CountVectorizer` converts a sample corpus into a term frequency matrix.

In [15]:
from sklearn.feature_extraction.text import CountVectorizer

docs = [
    "I love this movie",
    "This movie is terrible",
    "I love this"
]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(docs)

print("Vocabulary:", vectorizer.get_feature_names_out())
print("Count Vectors:\n", X.toarray())

Vocabulary: ['is' 'love' 'movie' 'terrible' 'this']
Count Vectors:
 [[0 1 1 0 1]
 [1 0 1 1 1]
 [0 1 0 0 1]]


#### Limitation of Bag of Words
The primary limitation of BoW is that it treats all words with equal importance based solely on their raw frequency. High-frequency words like "this", "is", "movie" appear across almost all documents and don't help differentiate between sentiments.

To overcome this, we use **TF-IDF**.

### 5.2 TF-IDF (Term Frequency-Inverse Document Frequency)

TF-IDF evaluates how important a word is to a document in a collection or corpus.

The TF-IDF weight is composed of two terms:
1. **Term Frequency (TF)**: How frequently a term occurs in a document.
   $$\text{TF}(t, d) = \frac{\text{Count of } t \text{ in } d}{\text{Total words in } d}$$
2. **Inverse Document Frequency (IDF)**: How common or rare a term is across all documents in the corpus.
   $$\text{IDF}(t, D) = \log\left(\frac{|D|}{1 + |\{d \in D : t \in d\}|}\right)$$

So, the TF-IDF weight is:
$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

Words that appear frequently in a specific document but rarely across the rest of the corpus get a high TF-IDF score. Words that appear in almost all documents get a very low score.

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

docs = [
    "The movie was so good!",
    "The movie was marvellous",
    "The movie was bulshit"
]

tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(docs)

print("Vocabulary:", tfidf_vectorizer.get_feature_names_out())
print("TF-IDF Vectors:\n", X_tfidf.toarray())

Vocabulary: ['bulshit' 'good' 'marvellous' 'movie' 'so' 'the' 'was']
TF-IDF Vectors:
 [[0.         0.57292883 0.         0.338381   0.57292883 0.338381
  0.338381  ]
 [0.         0.         0.69903033 0.41285857 0.         0.41285857
  0.41285857]
 [0.69903033 0.         0.         0.41285857 0.         0.41285857
  0.41285857]]


Let's also run a simple demonstration of TF-IDF on the previous `reviews` corpus to see how it compares to BoW.

In [17]:
reviews = [
    "good movie",
    "bad movie",
    "good love movie"
]

tfidf = TfidfVectorizer()
X_reviews = tfidf.fit_transform(reviews)

print("Features:", tfidf.get_feature_names_out())
print("TF-IDF Matrix:\n", X_reviews.toarray())

Features: ['bad' 'good' 'love' 'movie']
TF-IDF Matrix:
 [[0.         0.78980693 0.         0.61335554]
 [0.861037   0.         0.         0.50854232]
 [0.         0.54783215 0.72033345 0.42544054]]
